# Train Idlang: English ↔ Idoma in Google Colab
Run cells individually, from top to bottom. Select **Runtime → Change runtime type → T4 GPU** first. This notebook trains a new NLLB-600M checkpoint from the prepared Bible pairs. It does not guarantee translation accuracy: evaluate both directions and have an Idoma speaker review meaning, tone and fluency.

Use the `Idlang_Colab_Data_Prep.zip` supplied with this notebook. Training is full fine-tuning with Adafactor, batch size 1 and gradient checkpointing to reduce memory. It can take multiple Colab sessions. Checkpoints go to Drive. GPU availability and uninterrupted runtime are not guaranteed.

Do not use Run all: review data samples and length coverage before training, and review dev results before running the held-out test. No Hub login is required until the optional upload cell.

## 1. Install compatible libraries
Run this cell once per new runtime. Then use **Runtime → Restart session** before continuing with step 2. Transformers stays on version 4 to match the application runtime; do not upgrade to 5 for this notebook.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.1', 'datasets>=3,<5', 'accelerate>=1,<2',
    'sentencepiece', 'sacrebleu>=2,<3', 'huggingface_hub<1'])
print('Installation complete. Restart the session, then continue at step 2.')


## 2. Confirm GPU and connect Google Drive
Authorize Drive when Colab asks. Keep several GB free for model files, optimizer state, best/latest checkpoints and the final export; allow roughly 10 GB of headroom. Checkpoints are large, so saving can take time. Data is copied locally for fast reading.

In [ ]:
import os, json, hashlib, random, shutil, math, gc
from pathlib import Path
import numpy as np
import torch, transformers
from transformers import set_seed
from google.colab import drive
assert transformers.__version__ == '4.57.1', 'Restart after installation.'
assert torch.cuda.is_available(), 'Select a GPU runtime first.'
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory GB:', round(torch.cuda.get_device_properties(0).total_memory/2**30, 1))
print('PyTorch:', torch.__version__, '| Transformers:', transformers.__version__)
drive.mount('/content/drive')
set_seed(42)


## 3. Configure the run
Keep these settings for the first attempt. **RUN_NAME must change for a new experiment** (different length, dataset, learning rate, etc.). Reuse it only to resume the same run. Three epochs is an initial budget, not an accuracy guarantee.

`MAX_LENGTH=256` is a starting limit; step 5 measures real tokens and reports how many pairs it excludes. Whole overlength pairs are excluded within their existing splits, never chopped or reassigned. If substantial data is lost, choose a larger limit before training, considering GPU memory. Test scores will apply to the retained length-limited subset.

In [ ]:
BASE_MODEL = 'facebook/nllb-200-distilled-600M'
SRC_LANG, TGT_LANG, SEED_LANG = 'eng_Latn', 'idu_Latn', 'ibo_Latn'
RUN_NAME = 'idlang-bible-bidir-v1'
MAX_LENGTH = 256
EPOCHS = 3
LEARNING_RATE = 3e-5
BATCH_SIZE = 1
GRAD_ACCUM = 16
EVAL_SAVE_STEPS = 250
SEED = 42
PROJECT_DIR = Path('/content/drive/MyDrive/Idlang')
RUN_DIR = PROJECT_DIR / RUN_NAME
CHECKPOINT_DIR = RUN_DIR / 'checkpoints'
FINAL_DIR = RUN_DIR / 'final_model'
DATA_DIR = Path('/content/idlang_data')
for p in (PROJECT_DIR, RUN_DIR, CHECKPOINT_DIR, DATA_DIR):
    p.mkdir(parents=True, exist_ok=True)
print('Checkpoints:', CHECKPOINT_DIR)
print('Final model:', FINAL_DIR)


## 4. Upload the prepared ZIP and inspect data
Choose `Idlang_Colab_Data_Prep.zip` when prompted. The archive is also kept in Drive for reconnects. The loader supports `eng_Latn`/`idu_Latn` and the original `en`/`idu` keys. It checks both languages across all three splits. Review sample pairs with an Idoma speaker; matching references do not prove that every verse has the same segmentation or meaning.

In [ ]:
from google.colab import files
import io, zipfile
archive = PROJECT_DIR / 'Idlang_Colab_Data_Prep.zip'
if not archive.exists():
    uploaded = files.upload()
    candidates = [name for name in uploaded if name.endswith('.zip')]
    assert len(candidates) == 1, 'Select exactly one prepared ZIP.'
    archive.write_bytes(uploaded[candidates[0]])
with zipfile.ZipFile(archive) as z:
    for split in ('train', 'dev', 'test'):
        matches = [n for n in z.namelist() if n == 'prepared/' + split + '.jsonl' or n.endswith('/prepared/' + split + '.jsonl')]
        assert len(matches) == 1, f'Missing/ambiguous {split}.jsonl'
        (DATA_DIR / f'{split}.jsonl').write_bytes(z.read(matches[0]))
def read_rows(path):
    rows = []
    for line in path.read_text(encoding='utf-8').splitlines():
        if not line.strip(): continue
        r = json.loads(line)
        row = {'en': r.get('eng_Latn', r.get('en')), 'idu': r.get('idu_Latn', r.get('idu'))}
        assert all(isinstance(t, str) and t.strip() for t in row.values()), 'Empty/invalid text'
        assert row['en'].casefold() != row['idu'].casefold(), 'Echo pair'
        assert 'ụụ' not in row['idu'], 'Placeholder'
        rows.append(row)
    assert rows, f'Empty split: {path}'
    return rows
raw_splits = {s: read_rows(DATA_DIR / f'{s}.jsonl') for s in ('train','dev','test')}
for lang in ('en','idu'):
    sets = [{r[lang].casefold() for r in rows} for rows in raw_splits.values()]
    assert not any(sets[i] & sets[j] for i in range(3) for j in range(i)), f'{lang} leakage'
print({s:len(rows) for s,rows in raw_splits.items()})
for r in random.Random(SEED).sample(raw_splits['train'], 10):
    print('EN :', r['en'])
    print('IDU:', r['idu'])


## 5. Register Idoma and audit token lengths
Word counts are not NLLB token counts. This cell loads only the tokenizer, adds `idu_Latn`, measures untruncated sequences including language/end tokens, and excludes whole pairs above MAX_LENGTH. Review the printed retention rates before proceeding. Training/validation/test membership stays fixed.

In [ ]:
from transformers import AutoTokenizer
# The public base needs no HF token.
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, src_lang=SRC_LANG)
assert tokenizer.convert_tokens_to_ids(TGT_LANG) == tokenizer.unk_token_id
tokenizer.add_special_tokens({'additional_special_tokens': list(tokenizer.additional_special_tokens) + [TGT_LANG]})
idu_id = tokenizer.convert_tokens_to_ids(TGT_LANG)
assert idu_id != tokenizer.unk_token_id
for attr in ('lang_code_to_id', '_lang_code_to_id'):
    mapping = getattr(tokenizer, attr, None)
    if isinstance(mapping, dict): mapping[TGT_LANG] = idu_id

def count_tokens(rows, field, language):
    tokenizer.src_lang = language
    counts = []
    for start in range(0, len(rows), 256):
        result = tokenizer([r[field] for r in rows[start:start+256]], truncation=False,
                           padding=False, add_special_tokens=True)
        counts.extend(map(len, result['input_ids']))
    return counts

splits, length_report = {}, {}
for split, rows in raw_splits.items():
    en_len = count_tokens(rows, 'en', SRC_LANG)
    idu_len = count_tokens(rows, 'idu', TGT_LANG)
    keep = [i for i,(a,b) in enumerate(zip(en_len,idu_len)) if max(a,b) <= MAX_LENGTH]
    splits[split] = [rows[i] for i in keep]
    length_report[split] = {
        'original_pairs':len(rows), 'retained_pairs':len(keep),
        'retained_percent':100*len(keep)/len(rows),
        'english_tokens':{'mean':float(np.mean(en_len)), 'p95':float(np.percentile(en_len,95)), 'max':max(en_len)},
        'idoma_tokens':{'mean':float(np.mean(idu_len)), 'p95':float(np.percentile(idu_len,95)), 'max':max(idu_len)}}
    if split == 'train':
        for limit in (96,128,192,256,384,512):
            print(f'Train coverage at {limit} tokens: {100*sum(max(a,b)<=limit for a,b in zip(en_len,idu_len))/len(rows):.2f}%')
assert all(splits.values()), 'Length filter emptied a split.'
print(json.dumps(length_report, indent=2))
train_rows, dev_rows, test_rows = (splits[s] for s in ('train','dev','test'))
# Resume only with identical inputs/configuration.
manifest = {'base_model':BASE_MODEL, 'transformers':transformers.__version__,
    'input_sha256':{s:hashlib.sha256((DATA_DIR/f'{s}.jsonl').read_bytes()).hexdigest() for s in splits},
    'max_length':MAX_LENGTH, 'epochs':EPOCHS, 'learning_rate':LEARNING_RATE,
    'batch_size':BATCH_SIZE, 'gradient_accumulation':GRAD_ACCUM, 'seed':SEED,
    'optimizer':'adafactor', 'eval_save_steps':EVAL_SAVE_STEPS}
manifest_path = RUN_DIR / 'run_config.json'
if manifest_path.exists():
    assert json.loads(manifest_path.read_text()) == manifest, 'Changed experiment: use a NEW RUN_NAME.'
else:
    manifest_path.write_text(json.dumps(manifest, indent=2))
(RUN_DIR/'length_report.json').write_text(json.dumps(length_report, indent=2))


## 6. Build training examples in BOTH directions
Each retained training pair becomes two examples. Validation also includes both directions. Labels carry the correct target-language token. No truncation is used: assertions catch any sequence that exceeds the chosen limit.

In [ ]:
from datasets import Dataset

def directional_rows(rows, reverse=False):
    sl, tl = (TGT_LANG,SRC_LANG) if reverse else (SRC_LANG,TGT_LANG)
    sf, tf = ('idu','en') if reverse else ('en','idu')
    return [{'src':r[sf], 'tgt':r[tf], 'src_lang':sl, 'tgt_lang':tl} for r in rows]
def encode(batch):
    output = {'input_ids':[], 'attention_mask':[], 'labels':[]}
    for src,tgt,sl,tl in zip(batch['src'],batch['tgt'],batch['src_lang'],batch['tgt_lang']):
        tokenizer.src_lang, tokenizer.tgt_lang = sl, tl
        result = tokenizer(src, text_target=tgt, truncation=False)
        assert max(len(result['input_ids']), len(result['labels'])) <= MAX_LENGTH
        assert result['input_ids'][0] == tokenizer.convert_tokens_to_ids(sl)
        assert result['labels'][0] == tokenizer.convert_tokens_to_ids(tl)
        for k in output: output[k].append(result[k])
    return output
train_pairs = directional_rows(train_rows) + directional_rows(train_rows, reverse=True)
dev_pairs = directional_rows(dev_rows) + directional_rows(dev_rows, reverse=True)
train_ds = Dataset.from_list(train_pairs).map(encode, batched=True, remove_columns=list(train_pairs[0]))
dev_ds = Dataset.from_list(dev_pairs).map(encode, batched=True, remove_columns=list(dev_pairs[0]))
print('Training examples:',len(train_ds), '| validation examples:',len(dev_ds))
for i in (0, len(train_rows)):
    print('Source:',tokenizer.convert_ids_to_tokens(train_ds[i]['input_ids'][:1]),
          'target:',tokenizer.convert_ids_to_tokens(train_ds[i]['labels'][:1]))


## 7. Load and resize the model
This is the large download. A new Idoma embedding is initialized from the existing Igbo row as a starting point only; this does not make the languages equivalent. Real Idoma examples must train the new row and the rest of the model. Do not rerun this cell after training unless reconnecting and then resuming a checkpoint.

In [ ]:
from transformers import AutoModelForSeq2SeqLM
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
model.resize_token_embeddings(len(tokenizer), mean_resizing=False)
seed_id = tokenizer.convert_tokens_to_ids(SEED_LANG)
with torch.no_grad():
    model.get_input_embeddings().weight[idu_id].copy_(model.get_input_embeddings().weight[seed_id])
    if not model.config.tie_word_embeddings:
        model.get_output_embeddings().weight[idu_id].copy_(model.get_output_embeddings().weight[seed_id])
model.config.use_cache = False
model.config.forced_bos_token_id = idu_id
model.generation_config.forced_bos_token_id = idu_id
# A small sample detects whether learned weights differ from the starting base.
probe_name = next(n for n,p in model.named_parameters() if 'encoder.layers.0.fc1.weight' in n)
base_probe = dict(model.named_parameters())[probe_name][:8,:8].detach().cpu().clone()
print('Vocabulary:',len(tokenizer), '| Idoma token:',idu_id)


## 8. Configure training and run a forward-pass check
The notebook uses Adafactor to reduce optimizer memory, gradient checkpointing to reduce activation memory, and mixed precision. Label smoothing is zero, so NLLB computes its own decoder inputs without the old monkey-patch. Validation computes loss on both directions; generation is evaluated separately with the proper target token.

Every 250 optimizer steps, it evaluates and saves. Best and latest checkpoints are retained, and the best combined validation loss is loaded at completion. Early stopping allows four evaluations without improvement. These are starting settings, not proven optimal hyperparameters.

In [ ]:
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer, EarlyStoppingCallback
use_bf16 = torch.cuda.is_bf16_supported()
collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100)
args = Seq2SeqTrainingArguments(
    output_dir=str(CHECKPOINT_DIR), num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE, optim='adafactor', max_grad_norm=0.0,
    per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM, gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant':False},
    fp16=not use_bf16, bf16=use_bf16,
    warmup_ratio=0.05, weight_decay=0.01, label_smoothing_factor=0.0,
    logging_steps=25, logging_first_step=True,
    eval_strategy='steps', eval_steps=EVAL_SAVE_STEPS,
    save_strategy='steps', save_steps=EVAL_SAVE_STEPS, save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model='eval_loss', greater_is_better=False,
    prediction_loss_only=True, predict_with_generate=False,
    seed=SEED, data_seed=SEED, report_to=[],
)
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_ds,
    eval_dataset=dev_ds, data_collator=collator, processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)])
model.eval()
for i in (0,len(train_rows)):
    batch = {k:v.to(model.device) for k,v in collator([train_ds[i]]).items()}
    with torch.no_grad(), trainer.compute_loss_context_manager():
        result = model(**batch)
    assert torch.isfinite(result.loss), 'Non-finite initial loss.'
    print('Initial loss:',float(result.loss))
    del result, batch
gc.collect(); torch.cuda.empty_cache()
print('Forward checks passed. Next cell performs optimizer updates.')


## 9. Train or resume
Keep this cell running. It searches for the latest completed checkpoint. Checkpoints interrupted during writing do not have the notebook's completion marker and are skipped. Do not change the data/configuration when resuming. If the runtime disappears, reconnect, run steps 1–8 again with the same RUN_NAME, then rerun this cell.

A training-loss decrease is useful but does not establish translation quality. Watch validation loss, then inspect generated translations.

In [ ]:
from transformers import TrainerCallback
class MarkCompleteCheckpoint(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        folder = Path(args.output_dir) / f'checkpoint-{state.global_step}'
        (folder/'save_complete.json').write_text(json.dumps({'step':state.global_step}))
trainer.add_callback(MarkCompleteCheckpoint())
complete = [p for p in CHECKPOINT_DIR.glob('checkpoint-*')
            if (p/'save_complete.json').exists() and (p/'trainer_state.json').exists()]
resume = str(max(complete,key=lambda p:int(p.name.split('-')[-1]))) if complete else None
print('Resuming:',resume or 'new run')
result = trainer.train(resume_from_checkpoint=resume)
assert trainer.state.global_step > 0, 'No training steps completed.'
trainer.save_state()
(RUN_DIR/'training_metrics.json').write_text(json.dumps(result.metrics,indent=2))
(RUN_DIR/'loss_history.json').write_text(json.dumps(trainer.state.log_history,indent=2))
print('Best checkpoint:',trainer.state.best_model_checkpoint)
print('Best combined validation loss:',trainer.state.best_metric)
with torch.no_grad():
    drift = (model.get_input_embeddings().weight[idu_id] - model.get_input_embeddings().weight[seed_id]).abs().max().item()
    changed = not torch.equal(dict(model.named_parameters())[probe_name][:8,:8].detach().cpu(),base_probe)
print('Idoma vs seed row difference:',drift, '| sampled encoder weights changed:',changed)
assert drift > 0 and changed, 'Weights did not move as expected. Investigate before exporting.'
# Save promptly to Drive so later evaluation is recoverable.
model.config.use_cache = True
trainer.save_model(str(FINAL_DIR))
tokenizer.save_pretrained(str(FINAL_DIR))
print('Trained model and tokenizer saved:',FINAL_DIR)


## 10. Evaluate generation in both directions on DEVELOPMENT data
Each direction gets its own forced target-language token. chrF++ measures character/word overlap (0–100); it is not percent translation accuracy. BLEU and exact match are supplementary. Inspect hypotheses beside references. Do not tune the model against the test split.

This evaluates the full retained dev split, which can take time. Scores and per-example outputs are saved to Drive. An output hitting the generation limit without EOS is counted as capped; a high count means results are being cut short.

In [ ]:
import sacrebleu
from tqdm.auto import tqdm
model = trainer.model
model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()
@torch.inference_mode()
def translate_batch(texts, src_lang, tgt_lang):
    tokenizer.src_lang = src_lang
    batch = tokenizer(texts, padding=True, truncation=False, return_tensors='pt')
    if batch['input_ids'].shape[1] > MAX_LENGTH:
        raise ValueError(f'Input exceeds {MAX_LENGTH} tokens; shorten it or increase the supported limit.')
    batch = {k:v.to(model.device) for k,v in batch.items()}
    out = model.generate(**batch,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_lang),
        max_length=MAX_LENGTH+1, num_beams=4, do_sample=False)
    # Decoder-start is EOS for NLLB, so ignore position zero in the termination check.
    capped = [not bool((seq[1:]==tokenizer.eos_token_id).any()) for seq in out]
    return tokenizer.batch_decode(out, skip_special_tokens=True), capped

def score_direction(rows, reverse, split):
    sf,tf,sl,tl = ('idu','en',TGT_LANG,SRC_LANG) if reverse else ('en','idu',SRC_LANG,TGT_LANG)
    outputs=[]
    for start in tqdm(range(0,len(rows),2),desc=f'{split}: {sl} → {tl}'):
        selected=rows[start:start+2]
        hyp,capped=translate_batch([r[sf] for r in selected],sl,tl)
        outputs.extend({'source':r[sf],'reference':r[tf],'prediction':p,'generation_capped':c}
                       for r,p,c in zip(selected,hyp,capped))
    hypotheses=[r['prediction'] for r in outputs]
    refs=[r['reference'] for r in outputs]
    chrf=sacrebleu.metrics.CHRF(word_order=2)
    chrf_score=chrf.corpus_score(hypotheses,[refs])
    metrics={'pairs':len(rows),'chrF++':chrf_score.score,
        'chrf_signature':str(chrf.get_signature()),
        'BLEU':sacrebleu.corpus_bleu(hypotheses,[refs]).score,
        'exact_match_percent':100*sum(r['prediction'].strip()==r['reference'].strip() for r in outputs)/len(rows),
        'echo_percent':100*sum(r['prediction'].strip().casefold()==r['source'].strip().casefold() for r in outputs)/len(rows),
        'generation_capped_percent':100*sum(r['generation_capped'] for r in outputs)/len(rows)}
    stem=f'{split}_{sl}_to_{tl}'
    (RUN_DIR/f'{stem}_metrics.json').write_text(json.dumps(metrics,indent=2))
    with (RUN_DIR/f'{stem}_predictions.jsonl').open('w',encoding='utf-8') as f:
        for r in outputs: f.write(json.dumps(r,ensure_ascii=False)+'\n')
    print(stem,metrics)
    for r in random.Random(SEED).sample(outputs,min(10,len(outputs))): print(r)
    return metrics

dev_scores={
    'english_to_idoma':score_direction(dev_rows,False,'dev'),
    'idoma_to_english':score_direction(dev_rows,True,'dev'),
}


## 11. Test once after choosing the model
Run after you are satisfied with development results and have frozen the model/settings. These results describe the retained Bible subset only; they do not certify conversational translation or long verses. Both directions must be reviewed. If you later tune based on these test results, treat this split as development and obtain a new held-out test set.

In [ ]:
test_scores={
    'english_to_idoma':score_direction(test_rows,False,'test'),
    'idoma_to_english':score_direction(test_rows,True,'test'),
}
(RUN_DIR/'evaluation.json').write_text(json.dumps({'dev':dev_scores,'test':test_scores},indent=2))


## 12. Try your own sentences and verify saving
Replace the examples with held-out, human-reviewed everyday language. Keep a separate list with correct translations prepared by an Idoma speaker. Check omitted information, invented information, names/numbers, negation, tense, tone and fluency. Bible scores alone cannot answer whether the app handles conversation correctly.

In [ ]:
english_examples=['Good morning.', 'Where is the school?', 'I need some water.']
print(list(zip(english_examples,translate_batch(english_examples,SRC_LANG,TGT_LANG)[0])))
# Use your own reviewed Idoma sentences; this example comes from the supplied corpus.
idoma_examples=['Ohūtū ā, Ɔwɔicō le ɔkpáncō mɛ́mla ɛcɛ yá.']
print(list(zip(idoma_examples,translate_batch(idoma_examples,TGT_LANG,SRC_LANG)[0])))
reloaded_tokenizer=AutoTokenizer.from_pretrained(str(FINAL_DIR))
assert reloaded_tokenizer.convert_tokens_to_ids(TGT_LANG)==idu_id
for lang in (SRC_LANG,TGT_LANG):
    reloaded_tokenizer.src_lang=lang
    assert reloaded_tokenizer('probe')['input_ids'][0]==reloaded_tokenizer.convert_tokens_to_ids(lang)
shutil.copy2(RUN_DIR/'run_config.json',FINAL_DIR/'run_config.json')
shutil.copy2(RUN_DIR/'length_report.json',FINAL_DIR/'length_report.json')
if (RUN_DIR/'evaluation.json').exists(): shutil.copy2(RUN_DIR/'evaluation.json',FINAL_DIR/'evaluation.json')
card = f"""---
language:
- en
- idu
base_model: facebook/nllb-200-distilled-600M
pipeline_tag: translation
---
# Idlang English–Idoma Bible fine-tune
Full fine-tuning of NLLB-600M on {len(train_rows)} aligned Bible verse pairs, expanded to both directions. Idoma language token: idu_Latn. Seed 42. Maximum tokenized sequence length: {MAX_LENGTH}. See run_config.json, length_report.json and evaluation.json for settings and measured results.

Data: English KJV and Idoma Bible JSON supplied in maddox-bayn/idlang. Splits group repeated English or Idoma text together and stratify by length. This reduces exact-text leakage but does not exclude related passages in different books or nearby verses. Evaluation covers only pairs retained by the length filter. These are biblical-domain results, not a claim of general conversational fluency. Tone, dialect and translation accuracy require human review. Long inputs are outside the evaluated length range.
"""
(FINAL_DIR/'README.md').write_text(card,encoding='utf-8')
print('Final files:',sorted(p.name for p in FINAL_DIR.iterdir()))
print('All saved in Google Drive:',FINAL_DIR)


## 13. Optional: upload the reviewed model to Hugging Face
This cell does nothing until ENABLE_UPLOAD=True. Choose your own namespace and a NEW repository name so you can compare with the existing deployment. The login prompt accepts a write token; do not paste tokens into notebook source. Leave private while testing. The serving environment will need access to a private model. Only the final model folder is uploaded; training examples and full predictions stay in Drive.

In [ ]:
ENABLE_UPLOAD = False
HUB_REPO = 'YOUR_USERNAME/idlang-bible-bidirectional-v1'
if ENABLE_UPLOAD:
    assert 'YOUR_USERNAME' not in HUB_REPO, 'Set your own namespace.'
    assert (FINAL_DIR/'evaluation.json').exists(), 'Complete evaluation first.'
    from huggingface_hub import login, HfApi
    login()
    api=HfApi()
    api.create_repo(repo_id=HUB_REPO,private=True,exist_ok=True)
    api.upload_folder(repo_id=HUB_REPO,folder_path=str(FINAL_DIR))
    print('Uploaded:',HUB_REPO)
else:
    print('Upload disabled. The model is saved in Drive.')


## Recovery, troubleshooting and application integration
- **Disconnected:** run steps 1–9 with the same RUN_NAME and settings. Step 9 resumes the newest completed checkpoint. Colab's /content is temporary; Drive saves persist. A disconnect before the first checkpoint means starting again.
- **CUDA out of memory:** use batch size 1, restart the runtime to release old model objects, and run cleanly. If it persists at the chosen length, use a smaller length under a new RUN_NAME (and review data coverage), or a GPU with more memory. FP32 instead of FP16 needs more memory. The forward check cannot guarantee the backward pass will fit.
- **NaN/Inf loss:** do not export. Retry under a new RUN_NAME with a lower learning rate; if FP16 overflow persists, FP32 or BF16-capable hardware may be necessary.
- **Drive write failure:** check free space and connectivity. Do not treat a partially written checkpoint as complete.
- **Low loss but wrong language:** inspect both language-prefix assertions and pass the correct forced_bos_token_id at inference. Do not use ibo_Latn for Idoma.
- **Weak everyday translation:** add human-reviewed conversational pairs, deduplicate against all held-out data, and fine-tune/evaluate a new version. Repeating Bible epochs is not a substitute for domain coverage.
- **Serving:** set NMT_MODEL_ID to the reviewed model repo. For a private repo set HF_TOKEN in the server environment. Your current application has generation length settings too: review them against this notebook's MAX_LENGTH and test both directions directly through the Python service, then through Go. A dictionary hit in Go does not prove the neural model is working.

Sources: [Colab FAQ](https://research.google.com/colaboratory/faq.html), [Transformers 4.57.1 Trainer](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/trainer), [NLLB language controls](https://huggingface.co/docs/transformers/model_doc/nllb).

This notebook was checked for Python syntax and artifact structure. GPU training and Colab execution have not been performed in the preparation environment.